# Inspect the new response trace

This notebook runs the SDK against the local Narada development stack and displays both the existing trace fields and the new OpenAI-shaped flat trace. Start the backend, frontend, and extension development build first, then launch this notebook with:

```sh
set -a
source .env
set +a
uv run --with jupyter jupyter lab
```

In [ ]:
from __future__ import annotations

import asyncio
import json
import os
import socket
import tempfile
import urllib.request
from collections import defaultdict
from pathlib import Path
from typing import Any

from IPython.display import JSON, display
from narada import Agent, AgentKind, BrowserConfig, BrowserEnvironment, Span, Trace

## Configure the local services

The defaults expect the backend on port 8000, the frontend on port 3000, and the development extension build at `../caddie/src/google/chrome-extension/.output/chrome-mv3-dev`. The notebook uses `NARADA_API_KEY_DEV` when it is present, otherwise it falls back to `NARADA_API_KEY`.

In [2]:
DEV_EXTENSION_ID = "ijdopnjleolkjakldkjplfhniiohnccf"
WORKSPACE_ROOT = next(
    path
    for path in (Path.cwd(), *Path.cwd().parents)
    if (path / "narada-python-sdk").is_dir() and (path / "caddie").is_dir()
)
EXTENSION_PATH = (
    WORKSPACE_ROOT
    / "caddie"
    / "src"
    / "google"
    / "chrome-extension"
    / ".output"
    / "chrome-mv3-dev"
)


def find_chrome_for_testing() -> Path:
    cache_roots = (
        Path.home() / "Library" / "Caches" / "ms-playwright",
        Path.home() / ".cache" / "ms-playwright",
    )
    patterns = (
        "chromium-*/chrome-mac*/Google Chrome for Testing.app/Contents/MacOS/Google Chrome for Testing",
        "chromium-*/chrome-mac*/Chromium.app/Contents/MacOS/Chromium",
        "chromium-*/chrome-linux*/chrome",
        "chromium-*/chrome-win*/chrome.exe",
    )
    candidates = [
        executable
        for cache_root in cache_roots
        for pattern in patterns
        for executable in cache_root.glob(pattern)
        if executable.is_file() and os.access(executable, os.X_OK)
    ]
    if not candidates:
        raise RuntimeError("Run `uv run playwright install chromium` first.")
    return max(candidates, key=lambda path: path.stat().st_mtime)


def find_free_port() -> int:
    with socket.socket() as sock:
        sock.bind(("127.0.0.1", 0))
        return sock.getsockname()[1]


def get_loaded_extension_ids(cdp_port: int) -> set[str]:
    with urllib.request.urlopen(
        f"http://127.0.0.1:{cdp_port}/json/list",
        timeout=1,
    ) as response:
        targets = json.load(response)
    prefix = "chrome-extension://"
    return {
        url.removeprefix(prefix).partition("/")[0]
        for target in targets
        if isinstance(target, dict)
        and isinstance((url := target.get("url")), str)
        and url.startswith(prefix)
    }


async def wait_for_dev_extension(cdp_port: int) -> None:
    for _ in range(100):
        try:
            if DEV_EXTENSION_ID in await asyncio.to_thread(
                get_loaded_extension_ids,
                cdp_port,
            ):
                return
        except OSError:
            pass
        await asyncio.sleep(0.1)
    raise RuntimeError("The development extension did not load.")


if dev_api_key := os.getenv("NARADA_API_KEY_DEV"):
    os.environ["NARADA_API_KEY"] = dev_api_key
if "NARADA_API_KEY" not in os.environ:
    raise RuntimeError("Set NARADA_API_KEY_DEV or NARADA_API_KEY first.")
if not (EXTENSION_PATH / "manifest.json").is_file():
    raise RuntimeError("Build the development extension with `npm run local` first.")

os.environ.setdefault("NARADA_API_BASE_URL", "http://localhost:8000/fast/v2")
initialization_url = os.getenv(
    "NARADA_INITIALIZATION_URL",
    "http://localhost:3000/initialize",
)

browser_user_data = tempfile.TemporaryDirectory(prefix="narada-trace-demo-")
browser_config = BrowserConfig(
    executable_path=str(find_chrome_for_testing()),
    user_data_dir=browser_user_data.name,
    cdp_host="http://127.0.0.1",
    cdp_port=find_free_port(),
    initialization_url=initialization_url,
    extension_id=DEV_EXTENSION_ID,
)
browser_process = await asyncio.create_subprocess_exec(
    browser_config.executable_path,
    f"--user-data-dir={browser_config.user_data_dir}",
    f"--profile-directory={browser_config.profile_directory}",
    "--remote-debugging-address=127.0.0.1",
    f"--remote-debugging-port={browser_config.cdp_port}",
    f"--disable-extensions-except={EXTENSION_PATH}",
    f"--load-extension={EXTENSION_PATH}",
    "--no-default-browser-check",
    "--no-first-run",
    "about:blank",
    stdout=asyncio.subprocess.DEVNULL,
    stderr=asyncio.subprocess.DEVNULL,
)
await wait_for_dev_extension(browser_config.cdp_port)

environment = BrowserEnvironment(config=browser_config, attach_to_existing=True)
await environment.start()

RuntimeError: Set NARADA_API_KEY_DEV or NARADA_API_KEY first.

## Display helpers

The API returns a flat list. The tree below is reconstructed entirely from each span's `parent_id`.

In [ ]:
def exported_trace(records: list[Trace | Span[Any]] | None) -> list[dict[str, Any]]:
    return [record.model_dump(mode="json", by_alias=True) for record in records or []]


def span_label(span: Span[Any]) -> str:
    data = span.span_data
    return (
        getattr(data, "workflow_name", None)
        or getattr(data, "name", None)
        or getattr(data, "message", None)
        or data.type
    )


def show_trace(records: list[Trace | Span[Any]] | None) -> None:
    if not records:
        print("No response trace was available.")
        return

    trace = next(record for record in records if isinstance(record, Trace))
    spans = [record for record in records if isinstance(record, Span)]
    children: dict[str | None, list[Span[Any]]] = defaultdict(list)
    for span in spans:
        children[span.parent_id].append(span)

    print(f"{trace.name} ({trace.trace_id})")

    def print_children(parent_id: str | None, depth: int) -> None:
        for span in children[parent_id]:
            print(f"{'  ' * depth}- {span.span_data.type}: {span_label(span)}")
            print_children(span.span_id, depth + 1)

    print_children(None, 1)
    display(JSON(exported_trace(records), expanded=False))

## Direct Operator run

A direct call should produce one agent span with user-facing action spans beneath it.

In [ ]:
operator = Agent(environment=environment, kind=AgentKind.OPERATOR)
operator_response = await operator.run("Open example.com and describe the page title.")

print("Legacy action trace:")
display(operator_response.action_trace)
print("New response trace:")
show_trace(operator_response.trace)

## Agent Studio GUI workflow

Set `NARADA_DEMO_WORKFLOW` to an Agent Studio path such as `/owner/workflow-name`. A useful demo workflow contains an agent step, a loop, and a nested custom workflow.

In [ ]:
workflow_path = os.getenv("NARADA_DEMO_WORKFLOW")
if workflow_path:
    workflow = Agent(environment=environment, kind=workflow_path)
    workflow_response = await workflow.run("Run the observability demo.")

    print("Legacy workflow trace:")
    display(JSON(workflow_response.workflow_trace or {}, expanded=False))
    print("New response trace:")
    show_trace(workflow_response.trace)
else:
    print("Set NARADA_DEMO_WORKFLOW to run the workflow example.")

## Cleanup

In [ ]:
await environment.close(timeout=30)
if browser_process.returncode is None:
    browser_process.terminate()
    try:
        await asyncio.wait_for(browser_process.wait(), timeout=10)
    except TimeoutError:
        browser_process.kill()
        await browser_process.wait()
browser_user_data.cleanup()